# QUOPS with CUDA-Q

QUOPS measures the largest computationally relevant quantum circuit hardware can execute within a specified accuracy threshold and how quickly it can do so.


In [ ]:
# %pip install -r requirements.txt

import cudaq
import numpy as np
import pandas as pd
from time import perf_counter
from IPython.display import display
from src import *

cudaq.reset_target()
cudaq.set_target("density-matrix-cpu")
# cudaq.set_target("qpp-cpu")
# cudaq.set_target("nvidia", option="fp32") # gpu accelerated statevector simulator
# cudaq.set_target("nvidia", option="fp64") # Higher arithmetic precision and memory use.

# noise = None
noise = make_helios1_noise()
# noise = make_depolarizing_noise(p_1q=0.004, p_2q=0.02)

shots = 256
num_circuits = 100 # Independent circuits per shape; 16 cannot certify even ideal data with this bound.
num_mirrors = 1 # Fresh mirror randomizations for each forward circuit
bootstrap_resamples = 2000 # Estimate bootstrap spread as a diagnostic.
familywise_alpha = 0.05 # Family error budget for Hochberg across all declared shapes.
seed = 2026
rc_style = "pauli_frame" # Randomize reference circuits while preserving their ideal action.


## Circuit shape and mirror measurements

Layers of random CNOT pairings alternate with Pauli rotations, with axes and angles sampled independently and uniformly:

$$R_P(\theta)=e^{-i\theta P/2},\qquad P\in\{X,Y,Z\},\quad\theta\in[0,2\pi).$$

**Width** \(w\) counts qubits. **Paper depth** \(d\) counts elementary layers, so there are \(d/2\) CNOT/rotation pairs. \(\zeta\) is the active-qubit fraction in the final pair. **Forward size** counts qubit participations:

$$s=N_{R_P}+2N_{\rm CX}.$$

### BR, RR and REF

MCFE uses mirrors with known ideal output strings. With implementation \(g\), randomized reference compiler \(g_{\rm ref}\) and local-Clifford caps \(L,L'\) (rightmost acts first),

$$
\begin{aligned}
M_{\rm BR}&=L'g_{\rm ref}(C^\dagger)g(C)L,\\
M_{\rm RR}&=L'g_{\rm ref}(C^\dagger)g_{\rm ref}(C)L,\\
M_{\rm REF}&=L'L.
\end{aligned}
$$

BR probes the circuit; RR and REF estimate reference and cap/readout contributions. If \(\hat h_k\) is the fraction of shots at Hamming distance \(k\) from a mirror's target, its effective polarization is

$$\hat\lambda=\frac{\sum_{k=0}^{w}(-1/2)^k\hat h_k-4^{-w}}{1-4^{-w}}.$$



In [ ]:
widths = (3, 4)
depths = (4, 6)
zeta = 1.0 # Active-qubit fraction in the final pair.

# Keep only shapes inside the paper's utility cone: width**2 <= size <= width**3.
shapes = default_shape_schedule([{"width": w, "depth": d, "zeta": zeta} for w in widths for d in depths if in_utility_cone(w, shape_size(w, d, zeta))])

rng = np.random.default_rng(seed)
cudaq.set_random_seed(seed)

# Retain each circuit/mirror observation and update one live table in place.
rows = []
measurements = []
data = pd.DataFrame(rows)
live_table = display(data, display_id=True)

for shape in shapes:

    width, depth, size = shape["width"], shape["depth"], shape["size"]

    for circuit_index in range(num_circuits):

        # Draw the CNOT pairings, active subset, rotation axes and angles.
        C = sample_quops_circuit(width, depth, shape["zeta"], seed=rng)
        C_inverse = adjoint_C(C) # Reverse this exact sampled circuit.

        # Reuse C across its mirrors; independently drawn C circuits define the bootstrap clusters.
        for mirror_index in range(num_mirrors):

            # Independent caps give each mirror a known ideal target.
            br_caps = sample_mcfe_caps(width, seed=rng)
            L_br, L_prime_br, target_bits_br = br_caps.initial, br_caps.final, br_caps.target

            rr_caps = sample_mcfe_caps(width, seed=rng)
            L_rr, L_prime_rr, target_bits_rr = rr_caps.initial, rr_caps.final, rr_caps.target

            ref_caps = sample_mcfe_caps(width, seed=rng)
            L_ref, L_prime_ref, target_bits_ref = ref_caps.initial, ref_caps.final, ref_caps.target

            # Randomize each reference independently, preserving its ideal operation.
            rc_C_inverse_br = randomized_compilation(C_inverse, seed=rng, style=rc_style)
            rc_C_rr = randomized_compilation(C, seed=rng, style=rc_style)
            rc_C_inverse_rr = randomized_compilation(C_inverse, seed=rng, style=rc_style)

            # Time the BR and RR sampling calls for the simulator-rate proxy.
            start_time = perf_counter()
            br_counts = cudaq.sample(BR_kernel, width, L_br, C, rc_C_inverse_br, L_prime_br, shots_count=shots, noise_model=noise)
            rr_counts = cudaq.sample(RR_kernel, width, L_rr, rc_C_rr, rc_C_inverse_rr, L_prime_rr, shots_count=shots, noise_model=noise)
            br_rr_seconds = perf_counter() - start_time

            # Collect fresh independent REF data for each observation.
            ref_counts = cudaq.sample(REF_kernel, width, L_ref, L_prime_ref, shots_count=shots, noise_model=noise)

            # Convert all Hamming distances from the known target into each mirror's effective polarization.
            lambda_br = effective_polarization_from_counts(br_counts, width, target_bits_br)
            lambda_rr = effective_polarization_from_counts(rr_counts, width, target_bits_rr)
            lambda_ref = effective_polarization_from_counts(ref_counts, width, target_bits_ref)

            # Keep component estimates for ensemble inference and BR/RR shots plus timing for the rate.
            rows.append({"shape": f"w={width},s={size}", "width": width, "depth": depth, "size": size, "zeta": shape["zeta"], "circuit_index": circuit_index, "mirror_index": mirror_index, "lambda_br": lambda_br, "lambda_rr": lambda_rr, "lambda_ref": lambda_ref, "br_rr_seconds": br_rr_seconds, "m1_m2_shots": int(br_counts.get_total_shots() + rr_counts.get_total_shots()), "shots_br": int(br_counts.get_total_shots()), "shots_rr": int(rr_counts.get_total_shots()), "shots_ref": int(ref_counts.get_total_shots())})
            measurements.append({"width": width, "size": size, "circuit_index": circuit_index, "mirror_index": mirror_index, "counts": {"br": counts_dict(br_counts), "rr": counts_dict(rr_counts), "ref": counts_dict(ref_counts)}, "targets": {"br": target_bits_br, "rr": target_bits_rr, "ref": target_bits_ref}})
            data = pd.DataFrame(rows)
            live_table.update(data)

# Retain the complete acquisition and its raw evidence before inference.
evidence_path = save_notebook_evidence(data, measurements, shapes, {"shots": shots, "num_circuits": num_circuits, "num_mirrors": num_mirrors, "seed": seed, "bootstrap_resamples": bootstrap_resamples, "familywise_alpha": familywise_alpha, "multiplicity": "hochberg", "certification_method": "hoeffding", "rc_style": rc_style, "target": str(cudaq.get_target()), "noise": describe_noise(noise, max(widths))})
evidence_path


## Mean polarization and conservative shape tests

For each shape, average the BR, RR and REF observations and use the paper's ratio of means:

$$\hat\gamma_{w,s}=\frac{\bar\lambda_{\mathrm{BR}}}{\sqrt{\bar\lambda_{\mathrm{RR}}\bar\lambda_{\mathrm{REF}}}}.$$

The bootstrap resamples paired BR/RR circuit groups and independent REF observations. Its standard deviation is a diagnostic. Certification uses a finite-sample Hoeffding bound, with no normal approximation.

Each effective polarization lies in an interval of length \(L_w=3/[2(1-4^{-w})]\). For \(K\) independent circuit groups, \(N_{\rm REF}\) independent reference observations and error budget \(\alpha\), use

$$r_C=L_w\sqrt{\frac{\log(3/\alpha)}{2K}},\qquad r_R=L_w\sqrt{\frac{\log(3/\alpha)}{2N_{\rm REF}}}.$$

Define:

$$
\begin{aligned}
A_L &= \bar{\lambda}_{\mathrm{BR}} - r_C, \\
B_U &= \min\left(1, \bar{\lambda}_{\mathrm{RR}} + r_C\right), \\
R_U &= \min\left(1, \bar{\lambda}_{\mathrm{REF}} + r_R\right).
\end{aligned}
$$

When these values are positive, the individual lower bound is:

$$
\gamma_{\mathrm{lower}} = \frac{A_L}{\sqrt{B_U R_U}}.
$$

Otherwise the bound is uninformative. This requires positive population RR and REF means. Mirrors of the same original circuit do not increase \(K\).

The p-value inverts this bound against \(1/\sqrt e\). **Hochberg tests all declared shapes** with `familywise_alpha = 0.05`. Missing or invalid tests remain in the family with p-value 1 and cannot pass. Each shape has independent circuits and REF data here, as required for this use of Hochberg.

The plot shows the adjusted decisions. “Threshold not established” does not mean that the device is below threshold. The score is the largest certified size among the tested shapes. Individual lower bounds are not simultaneous bounds. No gated score tests are applied.

**Simulator throughput:** with summed BR/RR shots \(N\) and sampling-call time \(T\),

$$\hat\Omega_{\rm sim}(w,s)=\frac{2s\,\hat\gamma_{w,s}^{\,2}N}{T}.$$

The timer includes work inside BR/RR sampling calls. It excludes REF and work outside those calls. This is simulator throughput.

The sampling guarantee concerns the ratio of population means. Its interpretation as process polarization still depends on the MCFE model and reference compiler. The bound does not remove model bias.


In [ ]:
# Average the BR, RR and REF observations within each tested shape.
avg_lambdas_for_each_shape = calculate_mean_lambdas(data)

# Normalize the mean lambdas to estimate each shape's mean process polarization.
gamma_for_each_shape = calculate_gamma_hat_ws(avg_lambdas_for_each_shape)

# Retain paired-bootstrap spread as a diagnostic; certification uses the bounded sampling method.
sigma_for_each_shape = calculate_sigma(data, bootstrap_resamples, seed)

gamma_lower_for_each_shape = calculate_gamma_lower(gamma_for_each_shape, sigma_for_each_shape, alpha=familywise_alpha)

# Calculate one-sided p-values, then adjust across the full declared family.
p_value_for_each_shape = calculate_p_value(gamma_for_each_shape, sigma_for_each_shape)
hochberg_for_each_shape = hochberg_test(p_value_for_each_shape, alpha=familywise_alpha, declared_shapes=shapes)

# Calculate the simulator QUOPS/s proxy using summed BR/RR time and shots.
quops_rate_for_each_shape = calculate_quops_rate(data, gamma_for_each_shape)

# Plot every shape's Hochberg decision and its simulator rate side by side.
fig = plot_quops_scan(hochberg_for_each_shape, quops_rate_for_each_shape, title="QUOPS: tested shapes")


### Historical physical scan


<img src="plots/physical_capability_and_rate_scored.png" alt="Historical QUOPS simulation scan through 30 qubits, using the earlier analysis" width="1300">
